# NEXUS Alpha v3: Pro dashboard launcher

Runs the dark / light dashboard (`app_pro.py`) on the frozen **A3b** bundle. It uses the same engine and `bundle.json` as the notebook's own dashboard, so the numbers are the parity-tested ones.

1. **Runtime → Change runtime type → GPU** (a T4 is enough; CPU works but the first analysis is slow).
2. Run the two cells below. Open the link the second cell prints.
3. The first analysis loads the 13 models (1-2 minutes). Later ones take seconds.

`app_pro.py`, `screen_digitizer.py` and `screen_engine.py` (the screenshot reader) must be in the `app` folder next to `bundle.json` (it already is if you copied it there). The original `app.py` is not touched.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import subprocess, sys
APP = "/content/drive/MyDrive/antigravity_workspace/nexus_alpha_v3/artifacts_full_research/app"
PKGS = ["streamlit>=1.57", "plotly>=5.20", "timm>=1.0.15", "transformers>=4.48", "huggingface_hub>=0.26", "accelerate>=1.0", "sentencepiece>=0.2",
        "protobuf>=4.25", "ultralytics>=8.3.94,<8.4", "yfinance>=0.2.54", "joblib", "onnxruntime", "pillow", "scikit-learn==1.6.1"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], capture_output=True, text=True)
print("packages ok" if r.returncode == 0 else r.stderr[-2000:])
for pkg in ("rapidocr_onnxruntime", "rapidocr"):          # screenshot OCR is optional: without it, type the ticker
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True).returncode == 0:
        print("screenshot reader:", pkg); break
else:
    print("screenshot reader not installed: type the ticker instead of relying on screenshots")

In [ ]:
import os, time, urllib.request
from pathlib import Path
APPDIR = Path(APP)
for need in ("app_pro.py", "bundle.json", "nav3_core.py", "nav3_engine.py", "screen_reader.py", "screen_digitizer.py", "screen_engine.py", "flow_view.py", "arch_view.py"):
    assert (APPDIR / need).exists(), f"{need} is missing in {APPDIR}"
(APPDIR / ".streamlit").mkdir(exist_ok=True)
CONFIG = chr(10).join(["[theme]", 'base = "dark"', 'primaryColor = "#22d3ee"', "[server]", "maxUploadSize = 20", "[browser]", "gatherUsageStats = false", "[client]", 'toolbarMode = "minimal"'])
(APPDIR / ".streamlit" / "config.toml").write_text(CONFIG)
PORT = 8502
log = open("/content/streamlit_pro.log", "w")
proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(APPDIR / "app_pro.py"), "--server.port", str(PORT), "--server.headless", "true",
                         "--browser.gatherUsageStats", "false"], cwd=str(APPDIR), env={**os.environ, "NAV3_BUNDLE": str(APPDIR)}, stdout=log, stderr=log)
for _ in range(90):
    try:
        if urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=2).read().decode().strip() == "ok":
            break
    except Exception:
        time.sleep(1)
else:
    print("Streamlit did not start. Log:"); print(Path("/content/streamlit_pro.log").read_text()[-3000:])
from google.colab import output
print("Open the dashboard:", output.eval_js(f"google.colab.kernel.proxyPort({PORT})"))
output.serve_kernel_port_as_window(PORT)